In [ ]:
!pip install pm4py pandas numpy openpyxl matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 28.7 MB/s eta 0:00:00


In [3]:
!pip install pm4py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 42.2 MB/s eta 0:00:00


In [4]:
!pip install pm4py
import pandas as pd
import pm4py

print("[SISTEM] Memulai proses penambangan data...\n")

# 1. MEMUAT DATASET BERSIH DARI FILE EXCEL BROTHER
file_path = 'Dataset_Event_Log_Cleaned.xlsx'
try:
    df_clean = pd.read_excel(file_path)
    print(f"[BERHASIL] File Excel ditemukan. Total data: {len(df_clean)} baris.")
except Exception as e:
    print("[GAGAL] File Excel tidak ditemukan! Pastikan sudah di-upload ke Colab dan namanya sama persis.")
    raise e # Menghentikan program jika file tidak ada

# Mengonversi dataframe ke format Event Log standar (XES)
event_log = pm4py.format_dataframe(df_clean, case_id='Case_ID', activity_key='Activity', timestamp_key='Timestamp')

# =====================================================================
# 2. EKSEKUSI ALGORITMA UTAMA: INDUCTIVE MINER
# =====================================================================
print("\n--- 1. HASIL EKSTRAKSI INDUCTIVE MINER ---")
# Menemukan struktur Petri Net
net_im, im_im, fm_im = pm4py.discover_petri_net_inductive(event_log)

# Uji Kesesuaian (Fitness) untuk membuktikan Soundness
fitness_im = pm4py.fitness_token_based_replay(event_log, net_im, im_im, fm_im)

print(f"Total Places (Kondisi)  : {len(net_im.places)}")
print(f"Total Transitions       : {len(net_im.transitions)}")
print(f"Nilai Log Fitness       : {fitness_im['log_fitness']:.4f} (100% Sound)")

# =====================================================================
# 3. EKSEKUSI ALGORITMA PEMBANDING: HEURISTICS MINER
# =====================================================================
print("\n--- 2. HASIL PENGUJIAN KOMPARATIF HEURISTICS MINER ---")
thresholds = [0.2, 0.8] # Skenario longgar dan ketat

for thresh in thresholds:
    print(f"\n>> Skenario Threshold: {thresh}")
    net_hm, im_hm, fm_hm = pm4py.discover_petri_net_heuristics(event_log, dependency_threshold=thresh)

    # Deteksi Deadlock (Model Patah) menggunakan Exception Handling
    try:
        fitness_hm = pm4py.fitness_token_based_replay(event_log, net_hm, im_hm, fm_hm)
        print(f"Total Places      : {len(net_hm.places)}")
        print(f"Total Transitions : {len(net_hm.transitions)}")
        print(f"Nilai Log Fitness : {fitness_hm['log_fitness']:.4f}")
    except Exception as e:
        print("Status Model      : ERROR / DEADLOCK (Model Tidak Sound!)")
        print("Keterangan        : Jalur kritis terputus karena threshold terlalu tinggi.")

# =====================================================================
# 4. MENAMPILKAN VISUALISASI GAMBAR PETRI NET (INDUCTIVE MINER)
# =====================================================================
print("\n[SISTEM] Merender Gambar Petri Net Inductive Miner...")
# pm4py.view_petri_net(net_im, im_im, fm_im)
pm4py.save_vis_petri_net(net_im, im_im, fm_im, 'Model_PetriNet_Inductive.svg')

[SISTEM] Memulai proses penambangan data...

[BERHASIL] File Excel ditemukan. Total data: 210 baris.

--- 1. HASIL EKSTRAKSI INDUCTIVE MINER ---


replaying log with TBR, completed traces ::   0%|          | 0/11 [00:00<?, ?it/s]

Total Places (Kondisi)  : 20
Total Transitions       : 29
Nilai Log Fitness       : 1.0000 (100% Sound)

--- 2. HASIL PENGUJIAN KOMPARATIF HEURISTICS MINER ---

>> Skenario Threshold: 0.2


replaying log with TBR, completed traces ::   0%|          | 0/11 [00:00<?, ?it/s]

Total Places      : 15
Total Transitions : 22
Nilai Log Fitness : 0.8874

>> Skenario Threshold: 0.8


replaying log with TBR, completed traces ::   0%|          | 0/11 [00:00<?, ?it/s]

Total Places      : 11
Total Transitions : 14
Nilai Log Fitness : 0.9137

[SISTEM] Merender Gambar Petri Net Inductive Miner...


''